## Streaming Agent Responses

`agent.invoke()` waits until the entire run is complete before returning.
For a production chatbot or CLI tool, users see nothing until all reasoning
and tool calls finish — which can take seconds.

`agent.stream()` yields chunks **as each step completes**, giving real-time feedback.

**Topics covered:**
* `stream_mode="values"` — full state snapshot after every step
* `stream_mode="updates"` — only what changed per step (leaner output)
* `stream_mode="messages"` — token-by-token LLM output
* Streaming with tools
* Streaming with memory (thread_id)

In [ ]:
%run langchain_common.py

## Setup — a simple tool agent

We will use a calculator agent so we can see tool calls interleaved in the stream.

In [ ]:
from typing import Union

Number = Union[int, float]

@tool
def add(a: Number, b: Number) -> float:
    """Add two numbers."""
    return float(a + b)

@tool
def multiply(a: Number, b: Number) -> float:
    """Multiply two numbers."""
    return float(a * b)

agent = create_agent(llm_noreason, [add, multiply])

## 1. `invoke` — blocking, full response at the end

In [ ]:
result = agent.invoke({"messages": [HumanMessage(content="What is (3 + 4) * 5?")]})
print(result["messages"][-1].content)

## 2. `stream_mode="values"` — full state after every node

Each chunk is the **complete message list** at that point in the graph.
Useful when you want to see the entire conversation evolve step by step.

In [ ]:
print("=== stream_mode='values' ===")

for i, chunk in enumerate(agent.stream(
    {"messages": [HumanMessage(content="What is (3 + 4) * 5?")]},
    stream_mode="values",
)):
    last_msg = chunk["messages"][-1]
    print(f"\n--- Step {i + 1}: {type(last_msg).__name__} ---")
    last_msg.pretty_print()

## 3. `stream_mode="updates"` — only what changed per step

Each chunk is a `{node_name: delta}` dict — only the new messages added by that node.
More efficient than `values`; great for logging or displaying incremental progress.

In [ ]:
print("=== stream_mode='updates' ===")

for chunk in agent.stream(
    {"messages": [HumanMessage(content="What is (3 + 4) * 5?")]},
    stream_mode="updates",
):
    for node_name, delta in chunk.items():
        print(f"\n[Node: {node_name}]")
        for msg in delta.get("messages", []):
            msg.pretty_print()

## 4. `stream_mode="messages"` — token-by-token LLM output

This mode streams individual message chunks (tokens) from the LLM as they are generated.
Each chunk is a `(message_chunk, metadata)` tuple.
Tool messages are yielded whole (not tokenized).

In [ ]:
from langchain_core.messages import AIMessageChunk

print("=== stream_mode='messages' — token streaming ===")
print()

for msg_chunk, metadata in agent.stream(
    {"messages": [HumanMessage(content="What is (3 + 4) * 5? Explain each step.")]},
    stream_mode="messages",
):
    # AIMessageChunk tokens arrive one at a time — print without newline for live feel
    if isinstance(msg_chunk, AIMessageChunk) and msg_chunk.content:
        print(msg_chunk.content, end="", flush=True)

print()  # final newline

## 5. Streaming with memory (thread_id)

Streaming works exactly the same when the agent has a checkpointer.
Pass the `config` dict alongside the stream call.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

mem_agent = create_agent(llm_noreason, [add, multiply], checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "stream-demo"}}

In [ ]:
# Turn 1 — stream the first message
print("Turn 1:")
for msg_chunk, _ in mem_agent.stream(
    {"messages": [HumanMessage(content="My favourite number is 7.")]},
    config,
    stream_mode="messages",
):
    if isinstance(msg_chunk, AIMessageChunk) and msg_chunk.content:
        print(msg_chunk.content, end="", flush=True)
print()

# Turn 2 — agent remembers the favourite number
print("\nTurn 2:")
for msg_chunk, _ in mem_agent.stream(
    {"messages": [HumanMessage(content="What is my favourite number multiplied by 6?")]},
    config,
    stream_mode="messages",
):
    if isinstance(msg_chunk, AIMessageChunk) and msg_chunk.content:
        print(msg_chunk.content, end="", flush=True)
print()

## 6. Utility helper — `print_stream`

A reusable helper that pretty-prints streamed output. This is the pattern
you would use in a CLI chatbot or Jupyter widget.

In [ ]:
def print_stream(agent, question: str, config: dict | None = None):
    """Stream an agent response and print tokens as they arrive."""
    kwargs = {"stream_mode": "messages"}
    if config:
        kwargs["config"] = config

    for msg_chunk, _ in agent.stream({"messages": [HumanMessage(content=question)]}, **kwargs):
        if isinstance(msg_chunk, AIMessageChunk) and msg_chunk.content:
            print(msg_chunk.content, end="", flush=True)
    print()


print_stream(agent, "What is 12 + 99, then multiply by 3?")